# **Silver Layer Data Cleaning**

In [3]:
%pip install rapidfuzz

StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 9, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



#### **Import Uploaded Bronze Layer Data**

In [44]:
vendor_df = spark.read.table("bronze_vendor_export")
foundry_df = spark.read.table("bronze_foundry_reference")

vendor_pdf = vendor_df.toPandas()
foundry_ref_pdf = foundry_df.toPandas()


StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 51, Finished, Available, Finished, False)

#### **Clean the raw names before matching**

In [45]:
import re

def clean_name(name):
    if not name:
        return ""
    name = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', name)
    name = name.lower()
    name = re.sub(r'v\d+(\.\d+)?$', '', name)  # strip version suffixes like v2, v1.3
    name = re.sub(r'[-_]', ' ', name)           # underscores/hyphens -> spaces
    name = re.sub(r'\b(bold|italic|regular|light|medium|semibold|extrabold|black|thin|condense|reg|cond|med|lt|it|b|semi|extra|demi|ultra)\b', '', name) # strip style words
    name = re.sub(r'\s+', ' ', name).strip()
    return name

vendor_pdf["cleaned_name"] = vendor_pdf["raw_font_name"].apply(clean_name)
vendor_pdf[["raw_font_name", "cleaned_name"]].head(40)

StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 52, Finished, Available, Finished, False)

,raw_font_name,cleaned_name
0,Roboto-Bold-v2,roboto
1,ROBOTO CONDENSD,roboto condensd
2,Open Sans Italic,open sans
3,OpenSans-Regular,open sans
4,Montserrat SemiBold,montserrat
5,montserrat-black,montserrat
6,Lato Light v1.3,lato
7,Lato-Ligh,lato ligh
8,Poppins Medium,poppins
9,POPPINS-MED,poppins


#### **Fuzzy match against the canonical list**

In [47]:
import pandas as pd
from rapidfuzz import fuzz, process
from rapidfuzz.utils import default_process

canonical_families = foundry_ref_pdf["canonical_family"].tolist()

def match_font(cleaned_name):
    if not cleaned_name:
        return pd.Series([None, None, 0, "unresolved"])

    result = process.extractOne(
        cleaned_name,
        canonical_families,
        scorer=fuzz.WRatio,
        processor=default_process
    )
    
    if result is None:
        return pd.Series([None, None, 0, "unresolved"])

    match, score, _ = result

    if score >= 90:
        match_type = "high_confidence"
    elif score >= 75:
        match_type = "medium_confidence"
    else:
        return pd.Series([None, None, score, "unresolved"])

    foundry = foundry_ref_pdf.loc[foundry_ref_pdf["canonical_family"] == match, "canonical_foundry"].values[0]
    return pd.Series([match, foundry, score, match_type])

vendor_pdf[["matched_family", "matched_foundry", "confidence_score", "match_type"]] = vendor_pdf["cleaned_name"].apply(match_font)

StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 54, Finished, Available, Finished, False)

#### **Check results before writing anything**

In [48]:
display(vendor_pdf[["raw_font_name", "cleaned_name", "matched_family", "matched_foundry", "confidence_score", "match_type"]])

StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9ae76c4-af7e-4175-9a9d-1c5aef575bb4)

#### **Convert back to Spark and save as Silver Delta table**

In [49]:
silver_df = spark.createDataFrame(vendor_pdf, schema = list(vendor_pdf.columns))
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_matched_fonts")

StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 56, Finished, Available, Finished, False)

#### **Sanity-check the full table one more time before committing**

In [50]:
display(silver_df.orderBy("confidence_score"))

StatementMeta(, 1a784a3e-d7ac-4374-94b8-d9bd60dcf438, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 851be768-f5cf-40ee-ac02-78f5717eaa1e)